# Tribal Council TEST

In [1]:

import polars as pl
from inflect import pl_count_zero
import polars as pl
import social_groups.polars_columns as plc

from social_groups.analysis.defs.notebooks.definitions import register_materialization
from social_groups.analysis.polars_transformations.deserialize_experiment_configuration import deserialize_experiment_configuration
from social_groups.analysis.polars_transformations.make_group_constellation import parse_parameters, parse_model_family
from social_groups.polars_values import MODEL_NAME_TO_LETTER_MAPPING
from social_groups.reporting.group_reply import (
    GroupReplyAggregator,
    MajorityVote,
)
from social_groups.reporting.parsing import (
    AnswerComparer,
    AnswerOptions,
    AnswerParser,
)

/Users/philipp/Documents/Studium/Informatik/Masterthesis/Repository/.venv/lib/python3.11/site-packages/dagstermill/manager.py:398: BetaWarning: Class `Manager` is currently in beta, and may have breaking changes in minor version releases, with behavior changes in patch releases.
  MANAGER_FOR_NOTEBOOK_INSTANCE = Manager()
/Users/philipp/Documents/Studium/Informatik/Masterthesis/Repository/src/social_groups/analysis/notebook_assets.py:200: BetaWarning: Class `LocalFileCodeReference` is currently in beta, and may have breaking changes in minor version releases, with behavior changes in patch releases.
  dg.LocalFileCodeReference(
/Users/philipp/Documents/Studium/Informatik/Masterthesis/Repository/src/social_groups/analysis/notebook_assets.py:205: BetaWarning: Class `LocalFileCodeReference` is currently in beta, and may have breaking changes in minor version releases, with behavior changes in patch releases.
  dg.LocalFileCodeReference(
/Users/philipp/Documents/Studium/Informatik/Masterth

In [2]:
parser = AnswerParser(AnswerOptions.letters_A_to_J)
group_reply = GroupReplyAggregator(MajorityVote())
comparer = AnswerComparer(
    AnswerOptions.letters_A_to_J, triple_underscore_handling="wrong"
)

In [3]:
from social_groups.analysis.definitions import defs

combined_data: pl.DataFrame = defs().load_asset_value("combined_data")

/Users/philipp/Documents/Studium/Informatik/Masterthesis/Repository/.venv/lib/python3.11/site-packages/dagster/_config/pythonic_config/typing_utils.py:101: UserWarning: Field name "extension" in "PolarsParquetIOManager" shadows an attribute in parent "BasePolarsUPathIOManager"
  return super().__new__(cls, name, bases, namespaces, **kwargs)
2026-06-26 23:36:47 +0200 - dagster - DEBUG - system - Loading file from: /Users/philipp/Documents/Studium/Informatik/Masterthesis/Repository/results/analysis/dagster/combined_data.parquet using PolarsParquetIOManager...


In [4]:
tribal_council_frame = (
        combined_data.filter(
            pl.col("name").is_in(
                {"tribal_council-sweep"}
            )
        )
        .with_columns(
            deserialize_experiment_configuration(
                pl.col("experiment_configuration_json")
            ).alias("_experiment_configuration")
        )
        .with_columns(
            pl.col("_experiment_configuration")
            .struct.field("strategy")
            .struct.field("configuration")
            .struct.field("debate_agents")
            .list.eval(pl.element().struct.field("backend").struct.field("model_name"))
            .alias(plc.model_names),
            pl.col("_experiment_configuration")
            .struct.field("strategy")
            .struct.field("configuration")
            .struct.field("number_of_rounds")
            .alias("number_of_mad_rounds"),
            pl.col("_experiment_configuration")
            .struct.field("strategy")
            .struct.field("configuration")
            .struct.field("debate_agents")
            .list.eval(pl.element().struct.field("params").struct.field("temperature"))
            .list.unique()
            .list.item()
            .alias("temperature"),
            pl.col("_experiment_configuration")
            .struct.field("strategy")
            .struct.field("configuration")
            .struct.field("debate_agents")
            .list.eval(pl.element().struct.field("params").struct.field("top_p"))
            .list.unique()
            .list.item()
            .alias("top_p"),
        )
        .with_columns(
            pl.col(plc.model_names)
            .list.eval(parse_model_family(pl.element()))
            .list.unique()
            .list.item()
            .alias(plc.model_family),
            pl.col(plc.model_names)
            .list.eval(parse_parameters(pl.element()))
            .alias("model_parameters"),
            pl.col(plc.model_names)
            .list.eval(pl.element().replace(MODEL_NAME_TO_LETTER_MAPPING))
            .alias("sizes"),
        )
        .drop("_experiment_configuration")
    )

In [8]:
tribal_council_frame.filter(pl.col("answers_at_beginning").list.len() != 0).head()

id,run_id,message_ids,question_id,phoenix_span_id,phoenix_span_url,run_identifier,answers_at_beginning,answers_at_end,final_answer,used_input_tokens,used_output_tokens,experiment_id,experiment_configuration_json,execution_config_json,meta_info_json,name,original_question_id,data_connector,original_source,category,question,answer_options,answer_index,answer_string,model_names,number_of_mad_rounds,temperature,top_p,model_family,model_parameters,sizes
i64,i64,list[i64],i64,str,str,str,list[str],list[str],str,i64,i64,i64,str,str,str,str,i64,enum,str,str,str,list[str],i64,str,list[str],i64,f32,f32,str,list[f64],list[str]


In [10]:
from social_groups.trialrunner.decision_schemes.tribal_council import remove_answer
remove_answer("""Q: During a 6-month heating season, a homeowner expects to average $40 a month for fuel oil. He has purchased the following amounts to date: $37.50, $42.60, $39.80, $40.75, $44.10. What amount will he spend during the last month to maintain the expected average?\nOptions are:\n(A): $35.25\n(B): $36.50\n(C): $38.75\n(D): $46.00\n(E): $30.00\n(F): $41.20\n(G): $45.25\n(H): $40.00\n(I): $34.00\n(J): $32.90\n\n""", "a")

'Q: During a 6-month heating season, a homeowner expects to average $40 a month for fuel oil. He has purchased the following amounts to date: $37.50, $42.60, $39.80, $40.75, $44.10. What amount will he spend during the last month to maintain the expected average?\nOptions are:\n(A): $35.25\n(B): $36.50\n(C): $38.75\n(D): $46.00\n(E): $30.00\n(F): $41.20\n(G): $45.25\n(H): $40.00\n(I): $34.00\n(J): $32.90\n\n'